#Data Exploration

In [ ]:
# Import necessary libraries (pandas, numpy, matplotlib, seaborn, sklearn)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


In [ ]:
data = pd.read_csv('/content/Health_10.csv')
Health = pd.DataFrame(data)
Health = Health.drop(['Unnamed: 0', 'subject'] ,axis = 1)
Health

In [ ]:
Health.isnull().sum()

In [ ]:
plt.figure(figsize=(15,15))
sns.heatmap(Health.corr(),annot=True)

#PreProcessing Data

In [ ]:
#Extracting Independent and dependent Variable
x= Health.iloc[:, :-1].values
y= Health.iloc[:, -1].values

# Splitting the dataset into training and test set.

# CROSS-VALIDATION ---> Holdout Method
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test= train_test_split(x, y, test_size= 0.25, random_state=0)

#feature Scaling
from sklearn.preprocessing import StandardScaler
st_x= StandardScaler()

x_train= st_x.fit_transform(x_train)
x_test= st_x.transform(x_test)


#KNN

##Model Training

#### n_neighbors: To define the required neighbors of the algorithm. Usually, it takes 5.

#### metric='minkowski': This is the default parameter and it decides the distance between the points.

#### p=2: It is equivalent to the standard Euclidean metric. -> Euclidean distance.




In [ ]:
#Fitting K-NN classifier to the training set
from sklearn.neighbors import KNeighborsClassifier
classifier= KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2 )
classifier.fit(x_train, y_train)

In [ ]:
#Predicting the test set result
y_pred_KNN= classifier.predict(x_test)

##Evaluation

####Confusion_matrix

In [ ]:
#Creating the Confusion matrix
from sklearn.metrics import confusion_matrix
cm_KNN = confusion_matrix(y_test, y_pred_KNN)
print(cm_KNN)


####Classification report

In [ ]:
from sklearn.metrics import classification_report
report_KNN = classification_report(y_test, y_pred_KNN)
print(report_KNN)

#Linear Regression

## Model Training

In [ ]:
from sklearn import linear_model as LM
# Here, we will create linear regression object
reg1 = LM.LinearRegression()

# Now, we will train the model by using the training sets
reg1.fit(x_train, y_train)

## Evaluation with Mean Squared Error (MSE)

In [ ]:
from sklearn.metrics import mean_squared_error
y_pred_LM = reg1.predict(x_test)
MSE = mean_squared_error(y_test,y_pred_LM)
MSE

#Support Vector Machine (SVM)

##Model Training

In [ ]:
#Import svm model
from sklearn import svm

#Create a svm Classifier
clf = svm.SVC(kernel='rbf') # rbf Kernel

#Train the model using the training sets
clf.fit(x_train, y_train)

#Predict the response for test dataset
y_pred_SVM = clf.predict(x_test)


##Evaluation

###Confusion matrix

In [ ]:
cm_SVM = confusion_matrix(y_test, y_pred_SVM)
print(cm_SVM)

###Classification report

In [ ]:
report_SVM = classification_report(y_test, y_pred_SVM)
print(report_SVM)

#Neural Network

##Model Train

In [ ]:
Health['Activity'].unique()

In [ ]:
import tensorflow as tf

model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(12,)),  # input layer
    tf.keras.layers.Dense(units=30, activation="sigmoid"),  # hidden layer 1
    tf.keras.layers.Dense(units=64, activation="relu"),  # hidden layer 2
    tf.keras.layers.Dense(units=1, activation='sigmoid')  # output layer with 1 unit
])

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)
model.compile(loss="binary_crossentropy", metrics="accuracy", optimizer=optimizer)

In [ ]:
model.summary()

In [ ]:
history = model.fit(x_train, y_train, validation_data=(x_test, y_test), epochs=100)

In [ ]:
def visualize_performance(n_epochs, acc, val_acc, loss, val_loss):
  %matplotlib inline
  epochs = np.arange(n_epochs)
  acc = np.array(acc)
  val_acc = np.array(val_acc)
  loss = np.array(loss)
  val_loss = np.array(val_loss)
  plt.plot(epochs, acc*100, 'r', label='Training accuracy')
  plt.plot(epochs, val_acc*100, 'b', label='Validation accuracy')
  plt.scatter(epochs[val_acc.argmax()], val_acc.max()*100, color='green', s=70)
  plt.title('Training and validation accuracy')
  plt.legend()
  plt.figure()

  plt.plot(epochs, loss, 'r', label='Training Loss')
  plt.plot(epochs, val_loss, 'b', label='Validation Loss')
  plt.scatter(epochs[val_loss.argmin()], val_loss.min(), color='green', s=70)
  plt.title('Training and validation loss')
  plt.legend()

  plt.show()

In [ ]:
visualize_performance(100, history.history["accuracy"], history.history["val_accuracy"], history.history["loss"], history.history["val_loss"])

In [ ]:
# predict probabilities for test set
y_pred_NN = model.predict(x_test)

##Evaluation

###Confusion matrix Neural Network

In [ ]:
cm_NN= confusion_matrix(y_test, y_pred_NN)
print(cm_NN)

###Classification Report

In [ ]:
report_NN = classification_report(y_test, y_pred_NN)
print(report_NN)

#                                                     Logistic Regression

##Model Training

In [ ]:
from sklearn.linear_model import LogisticRegression

clf_LOG = LogisticRegression()
clf_LOG.fit(x_train,y_train)

In [ ]:
y_pred_LOG = clf_LOG.predict(x_test)


##Evaluation

### Confusion Matrix

In [ ]:
cm_LOG = confusion_matrix(y_test, y_pred_LOG)
cm_LOG

###Classification Report

In [ ]:
report_LOG = classification_report(y_test, y_pred_LOG)
print(report_LOG)

# Comparison of performance between different models

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, mean_squared_error

# Create list to store the performance metrics
performance_metrics = []

# List of models
models = ['KNN', 'SVM', 'Logistic Regression', 'Neural Network', 'Linear Regression']

# List of predictions
predictions = [y_pred_KNN, y_pred_SVM, y_pred_LOG, y_pred_NN, y_pred_LM]

for model, prediction in zip(models, predictions):
    if model == 'Linear Regression':
        # Use MSE for Linear Regression
        mse = mean_squared_error(y_test, prediction)
        performance_metrics.append({'Model': model, 'MSE': mse})
    else:
        accuracy = accuracy_score(y_test, prediction)
        precision = precision_score(y_test, prediction, average='weighted')
        recall = recall_score(y_test, prediction, average='weighted')
        f1 = f1_score(y_test, prediction, average='weighted')
        performance_metrics.append({'Model': model, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1 Score': f1})




In [ ]:
# Convert the list to a DataFrame
performance_metrics_df = pd.DataFrame(performance_metrics)
performance_metrics_df = performance_metrics_df.fillna('-')
performance_metrics_df